# NLP — โจทย์แบบ "ตัดคำ / ป้ายต่อตัวอักษร" (Sequence Labeling)

ภาษาไทยไม่มีเว้นวรรคระหว่างคำ งานคือบอกว่า "ตรงไหนคือจุดขึ้นคำใหม่"
มองเป็น: ตัวอักษรแต่ละตัว -> ป้าย 1 (ขึ้นคำใหม่) หรือ 0 (อยู่ในคำเดิม)

วิธีในโน้ตบุ๊กนี้:
- วิธีที่ 1 (ง่ายสุด มือใหม่ทำแค่นี้) = `PyThaiNLP deepcut` ตัดคำสำเร็จรูป ไม่ต้องเทรน
- วิธีที่ 2 (ไม่บังคับ) = `CRF` เทรนเองบน CPU ได้
- วิธีที่ 3 (ขั้นสูง ต้อง GPU) = `WangchanBERTa`


## เมื่อไหร่ใช้โน้ตบุ๊กนี้

ใช้เมื่อ output เป็น "ป้ายต่อหน่วยในลำดับ" เช่น ตัดคำ, NER (ดึงชื่อเฉพาะ), POS (ชนิดคำ)
ถ้า output เป็น "1 ป้ายต่อทั้งประโยค" (เช่น บวก/ลบ) -> ไปใช้ `text_classification.ipynb`

ต้องแก้ (`# << แก้`): ชื่อ competition, ชื่อไฟล์/คอลัมน์ข้อความ, ตัวคั่นคำในไฟล์เฉลย, รูปแบบ id ใน sample

## ขั้นที่ 1 — ติดตั้ง

In [ ]:
!pip -q install pythainlp deepcut attacut tensorflow python-crfsuite scikit-learn pandas numpy
!pip -q install transformers datasets torch   # วิธีที่ 3 เท่านั้น

## ขั้นที่ 2 — ตั้งค่า Kaggle แล้วดาวน์โหลดข้อมูล

ต้องมีไฟล์ `kaggle.json` ก่อน วิธีได้มา: เข้า kaggle.com -> กดรูปโปรไฟล์ -> Settings -> Account -> Create New Token
จะได้ไฟล์ `kaggle.json` หน้าตา `{"username":"...","key":"..."}`

- ถ้ารันบน `Kaggle Notebook`: ข้อมูลอยู่ที่ `/kaggle/input/...` แล้ว ไม่ต้องทำอะไร รันผ่านได้เลย
- ถ้ารันบน `Google Colab / เครื่องตัวเอง`: เอา username กับ key ใส่ในเซลล์ข้างล่าง (จุด `# << แก้`)

In [ ]:
import os, glob, subprocess

COMP = "super-ai-engineer-ss-6-word-segmentation"      # << แก้ตรงนี้: slug ของ competition คือคำท้าย URL เช่น kaggle.com/competitions/titanic -> ใส่ "titanic"
KAGGLE_USERNAME = ""   # << แก้ (เฉพาะ Colab/เครื่องตัวเอง) เช่น "peetwan1"  (บน Kaggle Notebook เว้นว่างได้)
KAGGLE_KEY      = ""   # << แก้ (เฉพาะ Colab/เครื่องตัวเอง) เช่น "0a1b2c3d4e5f6a7b8c9d..." (คีย์ยาว ~32 ตัวจาก kaggle.json)

def get_data(comp):
    # ถ้าอยู่บน Kaggle ข้อมูลถูกต่อไว้ให้แล้ว
    kpath = f"/kaggle/input/{comp}"
    if os.path.isdir(kpath):
        print("อยู่บน Kaggle แล้ว ข้อมูลอยู่ที่", kpath); return kpath
    # ถ้าไม่ใช่ Kaggle: เขียน token แล้วโหลดด้วย kaggle CLI
    if KAGGLE_USERNAME and KAGGLE_KEY:
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        open(os.path.expanduser("~/.kaggle/kaggle.json"), "w").write(
            '{"username":"%s","key":"%s"}' % (KAGGLE_USERNAME, KAGGLE_KEY))
        os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    os.makedirs("data", exist_ok=True)
    subprocess.run(["kaggle","competitions","download","-c",comp,"-p","data"], check=True)
    for z in glob.glob("data/*.zip"):
        subprocess.run(["unzip","-o","-q",z,"-d","data"], check=True)
    return "data"

DATA_DIR = get_data(COMP)
print("ไฟล์ที่ดาวน์โหลดมา (ดูชื่อไฟล์/โฟลเดอร์ แล้วเอาไปแก้ในเซลล์ถัดไป):")
for p in sorted(glob.glob(os.path.join(DATA_DIR, "**"), recursive=True))[:40]:
    print("  ", p)

## ฟังก์ชันหลัก — แปลงไป-กลับระหว่าง "คำ" กับ "ป้ายต่อตัวอักษร" 

In [ ]:
def words_to_char_labels(words):     # ['สวัสดี','ครับ'] -> ('สวัสดีครับ',[1,0,0,0,0,0,1,0,0,0])
    text,labels=[],[]
    for w in words:
        for i,ch in enumerate(w): text.append(ch); labels.append(1 if i==0 else 0)
    return "".join(text), labels
def char_labels_to_words(text,labels):
    words,cur=[],""
    for ch,lab in zip(text,labels):
        if lab==1 and cur: words.append(cur); cur=""
        cur+=ch
    if cur: words.append(cur)
    return words
from sklearn.metrics import f1_score
# ฟังก์ชันไว้วัดคะแนนตัวเองตอนแบ่ง train ลองทดสอบ (ไม่ได้ใช้ตอนส่งจริง) 1=ขึ้นคำใหม่, 0=กลางคำ
def boundary_f1(yt,yp): return f1_score(yt,yp,average="binary",pos_label=1)

## ขั้นที่ 3 — โหลดข้อมูล + CONFIG

In [ ]:
import os, pandas as pd, numpy as np
TRAIN_CSV  = os.path.join(DATA_DIR,"train.csv")   # << แก้ถ้าไฟล์ไม่ได้ชื่อนี้ (ดูชื่อจริงจากผลเซลล์ดาวน์โหลด) เช่น "training.csv"
TEST_CSV   = os.path.join(DATA_DIR,"test.csv")
SAMPLE_SUB = os.path.join(DATA_DIR,"sample_submission.csv")
TEXT_COL   = "text"     # << แก้: ชื่อคอลัมน์ข้อความ ดูจาก train.head() เช่น "text" หรือ "sentence"
ID_COL     = "id"       # << แก้: ชื่อคอลัมน์ id ของประโยคใน test (เปิด sample.head() ดู เช่น "id" หรือ "sentence_id")
SEP        = "|"        # << แก้: ตัวคั่นคำในเฉลย ถ้าเฉลยเขียน "ผม|รัก|แมว" ใช้ "|"; ถ้าเว้นวรรค "ผม รัก แมว" ใช้ " "
train=pd.read_csv(TRAIN_CSV); test=pd.read_csv(TEST_CSV); sample=pd.read_csv(SAMPLE_SUB)
print("train คอลัมน์:",list(train.columns)); print("sample คอลัมน์:",list(sample.columns))
display(train.head()); display(sample.head())
def load_segmented(df):
    out=[]
    for s in df[TEXT_COL].astype(str):
        out.append(words_to_char_labels([w for w in s.split(SEP) if w!=""]))
    return out
train_pairs=load_segmented(train)
print("ตัวอย่าง:", train_pairs[0][0][:30], "->", train_pairs[0][1][:30])

## วิธีที่ 1 — PyThaiNLP สำเร็จรูป (ง่ายสุด ไม่ต้องเทรน)

ลอง engine `deepcut` (แม่น ใกล้ SOTA) หรือ `newmm` (เร็วมาก) เก็บอันที่คะแนนดีกว่า

In [ ]:
from pythainlp.tokenize import word_tokenize
def to_labels(text, engine="deepcut"):   # << แก้ engine ได้: deepcut / newmm / attacut
    words=word_tokenize(str(text), engine=engine, keep_whitespace=True)
    _,labels=words_to_char_labels(words)
    return labels[:len(text)] + [0]*max(0,len(text)-len(labels))
# sample_submission ต้องการ 1 แถวต่อ "1 ตัวอักษร" รูปแบบ id = <id ประโยค>_<ลำดับตัวอักษรเริ่มที่ 0>
# ตัวอย่าง: ประโยค id=5 มี 3 ตัวอักษร -> "5_0","5_1","5_2"  << เปิด sample.head() เทียบให้ตรงเป๊ะ
rows=[]
for _,r in test.iterrows():
    labels=to_labels(str(r[TEXT_COL]))
    if labels: labels[0]=1   # ตัวอักษรตัวแรกเป็นจุดขึ้นคำเสมอ (ถ้าข้อความว่างให้ข้าม กันโปรแกรมพัง)
    sid=r[ID_COL]            # ใช้ id จริงจากคอลัมน์ ไม่ใช่เลขแถว
    for ci,lab in enumerate(labels):
        rows.append({sample.columns[0]: f"{sid}_{ci}", sample.columns[1]: lab})  # << แก้รูปแบบ id ให้ตรง sample (เช่น ใช้ "-" แทน "_")
sub=pd.DataFrame(rows); sub.to_csv("submission.csv",index=False)
print("บันทึก submission.csv", sub.shape); display(sub.head())

## วิธีที่ 2 — CRF เทรนเอง (ไม่บังคับ, CPU ได้, คะแนนดีในโดเมน)

In [ ]:
import pycrfsuite
def feats(chars,i):
    n=len(chars); f={"bias":1.0,"c0":chars[i]}
    for d in (1,2,3):
        f[f"c-{d}"]=chars[i-d] if i-d>=0 else "BOS"; f[f"c+{d}"]=chars[i+d] if i+d<n else "EOS"
    f["is_digit"]=chars[i].isdigit(); f["is_space"]=chars[i]==" "
    return f
tr=pycrfsuite.Trainer(verbose=False)
for text,labels in train_pairs:
    ch=list(text); tr.append([feats(ch,i) for i in range(len(ch))],[str(l) for l in labels])
tr.set_params({"c1":0.1,"c2":0.1,"max_iterations":100,"feature.possible_transitions":True})
tr.train("ws.crfsuite")
tag=pycrfsuite.Tagger(); tag.open("ws.crfsuite")
rows=[]
for _,r in test.iterrows():
    ch=list(str(r[TEXT_COL])); labels=[int(t) for t in tag.tag([feats(ch,i) for i in range(len(ch))])]
    if labels: labels[0]=1   # กันกรณีข้อความว่าง
    sid=r[ID_COL]
    for ci,lab in enumerate(labels):
        rows.append({sample.columns[0]: f"{sid}_{ci}", sample.columns[1]: lab})  # << แก้รูปแบบ id ให้ตรง sample
pd.DataFrame(rows).to_csv("submission_crf.csv",index=False); print("บันทึก submission_crf.csv")

## วิธีที่ 3 — WangchanBERTa (ขั้นสูง ต้อง GPU, ไม่บังคับ)

ดูโค้ดเต็มใน reference_cheatsheet.md หัวข้อ D.3 (token classification ด้วยโมเดลภาษาไทย)
มือใหม่ข้ามได้ ใช้วิธีที่ 1 ก็พอแล้วสำหรับการทำคะแนน

## ขั้นสุดท้าย — ส่ง submission ขึ้น Kaggle

เช็คก่อนส่งทุกครั้ง: ไฟล์ submission ต้องมีชื่อคอลัมน์ + จำนวนแถว ตรงกับ `sample_submission.csv` เป๊ะ ๆ
- บน Kaggle Notebook: กดปุ่ม `Submit` ที่แถบขวา (ง่ายสุด) หรือใช้คำสั่งข้างล่าง
- บน Colab/เครื่อง: รันเซลล์ข้างล่าง (ต้องตั้ง token แล้ว)

In [ ]:
FILE = "submission.csv"   # << แก้เป็นชื่อไฟล์ที่คะแนนดีสุด
!kaggle competitions submit -c {COMP} -f {FILE} -m "deepcut word segmentation"
!kaggle competitions submissions -c {COMP} | head